# MNIST Smoothing Radius: Sensitivity vs Case Align

This notebook sweeps Gaussian attribution smoothing radius from `0.1` to `2.0` and measures:

- Captum `sensitivity_max`
- Case Align like-neighbour `S_plus`

Both problem-space and solution-space distances use cosine distance, and Case Align uses `k=25`.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import torch
import torch.nn.functional as F
from captum.attr import DeepLift, IntegratedGradients
from captum.metrics import sensitivity_max
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

current_dir = Path.cwd()
if current_dir.name == "experiments":
    SRC_DIR = current_dir.parent
elif (current_dir / "src").exists():
    SRC_DIR = current_dir / "src"
else:
    SRC_DIR = current_dir

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from case_align.metrics import safe_normalise_rows
from explainers.lrp import LRP
from train_mnist_model import MNISTNet, set_seed

MODEL_PATH = SRC_DIR / "models" / "mnist" / "mnist_best_model.pt"
ARTIFACT_PATH = SRC_DIR / "explanations" / "mnist" / "mnist_explanations.pt"
DATA_DIR = SRC_DIR / "data"
OUTPUT_DIR = SRC_DIR / "results" / "mnist_smoothing_radius_sensitivity_vs_case_align_cosine_k25"

# Kept intentionally small for runtime: IG only.
METHODS = ["ig"]

K = 25
PROBLEM_METRIC = "cosine"
SOLUTION_METRIC = "cosine"
SMOOTHING_RADII = np.round(np.linspace(0.1, 2.0, 30), 2)

PERTURB_RADIUS = 0.1
N_PERTURB_SAMPLES = 5
RETRIEVAL_BATCH_SIZE = 256
MAX_QUERY_SAMPLES = 30
SEED = 42

METHOD_LABELS = {"ig": "Integrated Gradients", "dl": "DeepLift", "lrp": "LRP"}

print(f"SRC_DIR: {SRC_DIR}")
print(f"Artifact: {ARTIFACT_PATH}")
print(f"Model: {MODEL_PATH}")
print(f"k={K}, problem_metric={PROBLEM_METRIC}, solution_metric={SOLUTION_METRIC}")
print(f"Smoothing radii: {SMOOTHING_RADII[0]} to {SMOOTHING_RADII[-1]} ({len(SMOOTHING_RADII)} values)")

SRC_DIR: /Users/craigpirie/Documents/PhD/AGREE_ICCBR/agree-iccbr/src
Artifact: /Users/craigpirie/Documents/PhD/AGREE_ICCBR/agree-iccbr/src/explanations/mnist/mnist_explanations.pt
Model: /Users/craigpirie/Documents/PhD/AGREE_ICCBR/agree-iccbr/src/models/mnist/mnist_best_model.pt
k=25, problem_metric=cosine, solution_metric=cosine
Smoothing radii: 0.1 to 2.0 (30 values)


In [2]:
def _configure_ssl_for_macos() -> None:
    import ssl
    ssl._create_default_https_context = ssl._create_unverified_context


def load_model(model_path: Path, device: torch.device) -> MNISTNet:
    if not model_path.exists():
        raise FileNotFoundError(f"Model not found: {model_path}")

    checkpoint = torch.load(model_path, map_location=device)
    model = MNISTNet()
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
    elif isinstance(checkpoint, dict):
        model.load_state_dict(checkpoint)
    else:
        raise RuntimeError("Unsupported checkpoint format")

    model.to(device)
    model.eval()
    return model


def load_full_test_split(data_dir: Path):
    _configure_ssl_for_macos()
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)),
    ])
    test_dataset = datasets.MNIST(str(data_dir), train=False, download=True, transform=transform)
    loader = DataLoader(test_dataset, batch_size=512, shuffle=False, num_workers=0)

    image_chunks, label_chunks = [], []
    for images, labels in loader:
        image_chunks.append(images)
        label_chunks.append(labels)

    return torch.cat(image_chunks, dim=0).float(), torch.cat(label_chunks, dim=0).numpy().astype(int)


def predict_labels_for_images(model, images, batch_size, device):
    pred_chunks, conf_chunks = [], []
    with torch.no_grad():
        for start in range(0, len(images), batch_size):
            end = min(start + batch_size, len(images))
            logits = model(images[start:end].to(device))
            probs = torch.softmax(logits, dim=1)
            conf, preds = probs.max(dim=1)
            pred_chunks.append(preds.detach().cpu().numpy().astype(int))
            conf_chunks.append(conf.detach().cpu().numpy().astype(float))

    return np.concatenate(pred_chunks), np.concatenate(conf_chunks)


class MNISTLogitsWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        return self.model(x, return_logits=True)


def build_explainers(model, methods):
    explainers = {}
    logits_model = MNISTLogitsWrapper(model)
    for method in methods:
        if method == "ig":
            explainers[method] = IntegratedGradients(model)
        elif method == "dl":
            explainers[method] = DeepLift(logits_model)
        elif method == "lrp":
            explainers[method] = LRP(logits_model)
        else:
            raise ValueError(f"Unknown method: {method}")
    return explainers


def cosine_distance_context(matrix, epsilon=1e-8):
    matrix_raw = np.asarray(matrix, dtype=float)
    return {
        "matrix_raw": matrix_raw,
        "matrix_repr": safe_normalise_rows(matrix_raw, eps=epsilon),
        "epsilon": float(epsilon),
    }


def cosine_row_distances(query_vec, context):
    query_repr = safe_normalise_rows(np.asarray(query_vec, dtype=float)[None, :], eps=context["epsilon"])[0]
    similarity = context["matrix_repr"] @ query_repr
    return np.clip(1.0 - 0.5 * (similarity + 1.0), 0.0, 1.0)


def compute_case_align_like_only(query_index, query_label, retrieval_labels, problem_context, solution_context, k, epsilon=1e-8):
    query_problem = problem_context["matrix_raw"][query_index]
    dprob_all = cosine_row_distances(query_problem, problem_context)

    like_mask = retrieval_labels == int(query_label)
    like_mask[query_index] = False
    like_indices = np.where(like_mask)[0]
    if like_indices.size == 0:
        return np.nan, 0, False

    like_dists = dprob_all[like_indices]
    k_use = min(int(k), like_indices.size)
    neigh_indices = like_indices[np.argsort(like_dists)[:k_use]]
    dprob_neigh = dprob_all[neigh_indices]

    query_solution = solution_context["matrix_raw"][query_index]
    dsoln_all = cosine_row_distances(query_solution, solution_context)
    dsoln_neigh = dsoln_all[neigh_indices]

    denom = max(float(np.max(dsoln_all) - np.min(dsoln_all)), epsilon)
    align = 1.0 - (dsoln_neigh - float(np.min(dsoln_all))) / denom
    weights = 1.0 - dprob_neigh
    weight_sum = float(np.sum(weights))
    if weight_sum <= 0:
        return 0.0, int(k_use), True

    return float(np.sum(weights * align) / weight_sum), int(k_use), True


def gaussian_kernel2d(radius, device, dtype):
    sigma = float(radius)
    kernel_size = max(3, int(2 * np.ceil(3 * sigma) + 1))
    coords = torch.arange(kernel_size, device=device, dtype=dtype) - (kernel_size - 1) / 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    kernel = torch.outer(g, g)
    kernel = kernel / kernel.sum()
    return kernel.unsqueeze(0).unsqueeze(0)


def smooth_attributions(attrs, radius):
    kernel = gaussian_kernel2d(radius, attrs.device, attrs.dtype)
    kernel_size = kernel.shape[-1]
    channels = attrs.shape[1]
    kernel = kernel.expand(channels, 1, kernel_size, kernel_size)
    return F.conv2d(attrs, kernel, padding=kernel_size // 2, groups=channels)


def attribute_batch(explainer, method, images, targets, baseline, device):
    xb = images.to(device).clone().detach().requires_grad_(True)
    target_tensor = torch.as_tensor(targets, dtype=torch.long, device=device)
    if method in {"ig", "dl"}:
        return explainer.attribute(xb, baselines=baseline.to(device).expand_as(xb), target=target_tensor)
    return explainer.attribute(xb, target=target_tensor)


def compute_retrieval_attributions(explainer, method, images, pred_labels, baseline, batch_size, device):
    chunks = []
    n_batches = (len(images) + batch_size - 1) // batch_size
    for batch_idx, start in enumerate(range(0, len(images), batch_size), start=1):
        end = min(start + batch_size, len(images))
        attrs = attribute_batch(explainer, method, images[start:end], pred_labels[start:end], baseline, device)
        chunks.append(attrs.detach().cpu().float())
        if batch_idx == 1 or batch_idx == n_batches or batch_idx % max(n_batches // 10, 1) == 0:
            print(f"      attribution batches: {batch_idx}/{n_batches}")
    return torch.cat(chunks, dim=0)


def compute_sensitivity_for_radius(explainer, method, image, target, baseline, perturb_radius, n_perturb_samples, device, smoothing_radius):
    x = image.unsqueeze(0).to(device)
    b0 = baseline.to(device)

    def explain_func(inputs, target=None):
        x_in = inputs[0] if isinstance(inputs, tuple) else inputs
        target_tensor = target
        if method in {"ig", "dl"}:
            attrs = explainer.attribute(x_in, baselines=b0.expand_as(x_in), target=target_tensor)
        else:
            attrs = explainer.attribute(x_in, target=target_tensor)
        return smooth_attributions(attrs, radius=smoothing_radius)

    sens = sensitivity_max(
        explanation_func=explain_func,
        inputs=x,
        target=int(target),
        perturb_radius=perturb_radius,
        n_perturb_samples=n_perturb_samples,
    )
    return float(sens.detach().cpu().item())

In [3]:
set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if PROBLEM_METRIC != "cosine" or SOLUTION_METRIC != "cosine":
    raise ValueError("This notebook is configured for cosine in both problem and solution spaces.")
if K != 25:
    raise ValueError("This notebook is configured for k=25.")
if not ARTIFACT_PATH.exists():
    raise FileNotFoundError(f"Explanation artifact not found: {ARTIFACT_PATH}")
if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model checkpoint not found: {MODEL_PATH}")

artifact = torch.load(ARTIFACT_PATH, map_location="cpu")
artifact_methods = [m.lower() for m in artifact.get("methods", ["ig", "dl", "lrp"])]
methods_to_run = artifact_methods if METHODS is None else [m.lower() for m in METHODS]
methods_to_run = [m for m in methods_to_run if m in {"ig", "dl", "lrp"}]
if not methods_to_run:
    raise RuntimeError("No valid methods selected.")

query_images = artifact["images"].float()
query_labels = artifact["labels"].numpy().astype(int)
query_pred_labels = artifact["pred_labels"].numpy().astype(int)
query_confidences = artifact["confidences"].numpy().astype(float)
query_retrieval_indices = artifact["sample_indices"].numpy().astype(int)
baseline = artifact["baseline"].float()

if MAX_QUERY_SAMPLES is not None and len(query_images) > MAX_QUERY_SAMPLES:
    query_images = query_images[:MAX_QUERY_SAMPLES]
    query_labels = query_labels[:MAX_QUERY_SAMPLES]
    query_pred_labels = query_pred_labels[:MAX_QUERY_SAMPLES]
    query_confidences = query_confidences[:MAX_QUERY_SAMPLES]
    query_retrieval_indices = query_retrieval_indices[:MAX_QUERY_SAMPLES]

model = load_model(MODEL_PATH, device)
explainers = build_explainers(model, methods_to_run)

print("Loading full MNIST test split for retrieval...")
retrieval_images, retrieval_labels = load_full_test_split(DATA_DIR)
retrieval_pred_labels, retrieval_confidences = predict_labels_for_images(model, retrieval_images, RETRIEVAL_BATCH_SIZE, device)

if np.any(query_retrieval_indices < 0) or np.any(query_retrieval_indices >= len(retrieval_labels)):
    raise IndexError("Some artifact sample_indices are outside the retrieval pool.")

X_flat = retrieval_images.view(len(retrieval_images), -1).detach().cpu().numpy()
problem_context = cosine_distance_context(X_flat)

print(f"Methods: {methods_to_run}")
print(f"Query samples: {len(query_images)}")
print(f"Retrieval pool size: {len(retrieval_images)}")

Using device: cpu


Loading full MNIST test split for retrieval...


Methods: ['ig']
Query samples: 30
Retrieval pool size: 10000


## Run Smoothing-Radius Sweep

In [4]:
score_rows = []

for method in methods_to_run:
    print(f"\n=== {METHOD_LABELS.get(method, method.upper())} ===")
    print("  Computing unsmoothed retrieval attributions once...")
    raw_retrieval_attrs = compute_retrieval_attributions(
        explainer=explainers[method],
        method=method,
        images=retrieval_images,
        pred_labels=retrieval_pred_labels,
        baseline=baseline,
        batch_size=RETRIEVAL_BATCH_SIZE,
        device=device,
    )

    for radius in SMOOTHING_RADII:
        radius = float(radius)
        print(f"  Radius {radius:.2f}: Case Align + sensitivity")

        smoothed_attrs = smooth_attributions(raw_retrieval_attrs.to(device), radius=radius).detach().cpu()
        E_flat = smoothed_attrs.view(len(retrieval_images), -1).numpy()
        solution_context = cosine_distance_context(E_flat)

        for i in range(len(query_images)):
            retrieval_index = int(query_retrieval_indices[i])
            s_plus, like_count, has_like_neighbour = compute_case_align_like_only(
                query_index=retrieval_index,
                query_label=int(query_labels[i]),
                retrieval_labels=retrieval_labels,
                problem_context=problem_context,
                solution_context=solution_context,
                k=K,
            )

            sensitivity = compute_sensitivity_for_radius(
                explainer=explainers[method],
                method=method,
                image=query_images[i],
                target=int(query_pred_labels[i]),
                baseline=baseline,
                perturb_radius=PERTURB_RADIUS,
                n_perturb_samples=N_PERTURB_SAMPLES,
                device=device,
                smoothing_radius=radius,
            )

            score_rows.append({
                "method": method,
                "method_label": METHOD_LABELS.get(method, method.upper()),
                "smoothing_radius": radius,
                "sample_position": int(i),
                "original_test_index": retrieval_index,
                "true_label": int(query_labels[i]),
                "pred_label": int(query_pred_labels[i]),
                "confidence": float(query_confidences[i]),
                "case_align_like_count": int(like_count),
                "case_align_has_like_neighbour": bool(has_like_neighbour),
                "case_align_S_plus": float(s_plus) if has_like_neighbour else np.nan,
                "captum_sensitivity": float(sensitivity),
            })

scores_df = pd.DataFrame(score_rows)
summary_df = (
    scores_df.groupby(["method", "method_label", "smoothing_radius"], as_index=False)
    .agg(
        n_samples=("sample_position", "count"),
        mean_case_align_S_plus=("case_align_S_plus", "mean"),
        std_case_align_S_plus=("case_align_S_plus", "std"),
        mean_captum_sensitivity=("captum_sensitivity", "mean"),
        std_captum_sensitivity=("captum_sensitivity", "std"),
    )
)

print("\nSummary:")
display(summary_df.round(5))


=== Integrated Gradients ===
  Computing unsmoothed retrieval attributions once...


      attribution batches: 1/40


      attribution batches: 4/40


      attribution batches: 8/40


      attribution batches: 12/40


...

  Radius 1.87: Case Align + sensitivity


  Radius 1.93: Case Align + sensitivity


  Radius 2.00: Case Align + sensitivity



Summary:


,method,method_label,smoothing_radius,n_samples,mean_case_align_S_plus,std_case_align_S_plus,mean_captum_sensitivity,std_captum_sensitivity
0,ig,Integrated Gradients,0.10,30,0.53270,0.08207,0.56830,0.20146
1,ig,Integrated Gradients,0.17,30,0.53270,0.08207,0.56471,0.25621
2,ig,Integrated Gradients,0.23,30,0.53278,0.08207,0.58101,0.23641
3,ig,Integrated Gradients,0.30,30,0.53678,0.08224,0.57371,0.21152
4,ig,Integrated Gradients,0.36,30,0.55439,0.08296,0.54072,0.14294
5,ig,Integrated Gradients,0.43,30,0.59666,0.08460,0.50655,0.19462
6,ig,Integrated Gradients,0.49,30,0.64051,0.08606,0.46107,0.18856
7,ig,Integrated Gradients,0.56,30,0.68690,0.08575,0.40770,0.15197
8,ig,Integrated Gradients,0.62,30,0.71766,0.08449,0.35951,0.11779
9,ig,Integrated Gradients,0.69,30,0.74460,0.08251,0.31473,0.12010


## Plotly Line Graph

In [5]:
plot_df = summary_df.melt(
    id_vars=["method", "method_label", "smoothing_radius"],
    value_vars=["mean_case_align_S_plus", "mean_captum_sensitivity"],
    var_name="metric",
    value_name="mean_value",
)
plot_df["metric"] = plot_df["metric"].map({
    "mean_case_align_S_plus": "Case Align S_plus",
    "mean_captum_sensitivity": "Captum sensitivity_max",
})
plot_df["series"] = plot_df["method_label"] + " - " + plot_df["metric"]

fig = px.line(
    plot_df,
    x="smoothing_radius",
    y="mean_value",
    color="series",
    markers=True,
    title="MNIST smoothing radius: sensitivity vs Case Align (cosine, k=25)",
    labels={"smoothing_radius": "Smoothing radius", "mean_value": "Mean score", "series": "Metric"},
)
fig.update_layout(hovermode="x unified")
fig.show()

fig_by_metric = px.line(
    plot_df,
    x="smoothing_radius",
    y="mean_value",
    color="method_label",
    facet_row="metric",
    markers=True,
    title="MNIST smoothing radius sweep by metric (cosine, k=25)",
    labels={"smoothing_radius": "Smoothing radius", "mean_value": "Mean score", "method_label": "Explainer"},
)
fig_by_metric.update_yaxes(matches=None)
fig_by_metric.update_layout(hovermode="x unified")
fig_by_metric.show()

In [6]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

scores_path = OUTPUT_DIR / "mnist_smoothing_radius_scores_cosine_k25.csv"
summary_path = OUTPUT_DIR / "mnist_smoothing_radius_summary_cosine_k25.csv"
plot_long_path = OUTPUT_DIR / "mnist_smoothing_radius_plot_long_cosine_k25.csv"
config_path = OUTPUT_DIR / "mnist_smoothing_radius_config_cosine_k25.json"

scores_df.to_csv(scores_path, index=False)
summary_df.to_csv(summary_path, index=False)
plot_df.to_csv(plot_long_path, index=False)

with config_path.open("w", encoding="utf-8") as f:
    json.dump({
        "model_path": str(MODEL_PATH),
        "artifact_path": str(ARTIFACT_PATH),
        "data_dir": str(DATA_DIR),
        "methods": methods_to_run,
        "k": int(K),
        "problem_metric": PROBLEM_METRIC,
        "solution_metric": SOLUTION_METRIC,
        "smoothing_radii": [float(x) for x in SMOOTHING_RADII],
        "perturb_radius": float(PERTURB_RADIUS),
        "n_perturb_samples": int(N_PERTURB_SAMPLES),
        "retrieval_batch_size": int(RETRIEVAL_BATCH_SIZE),
        "max_query_samples": MAX_QUERY_SAMPLES,
        "n_query_samples": int(len(query_images)),
        "retrieval_pool_size": int(len(retrieval_images)),
    }, f, indent=2)

print("Saved outputs:")
print(f"- Scores: {scores_path}")
print(f"- Summary: {summary_path}")
print(f"- Plot data: {plot_long_path}")
print(f"- Config: {config_path}")

Saved outputs:
- Scores: /Users/craigpirie/Documents/PhD/AGREE_ICCBR/agree-iccbr/src/results/mnist_smoothing_radius_sensitivity_vs_case_align_cosine_k25/mnist_smoothing_radius_scores_cosine_k25.csv
- Summary: /Users/craigpirie/Documents/PhD/AGREE_ICCBR/agree-iccbr/src/results/mnist_smoothing_radius_sensitivity_vs_case_align_cosine_k25/mnist_smoothing_radius_summary_cosine_k25.csv
- Plot data: /Users/craigpirie/Documents/PhD/AGREE_ICCBR/agree-iccbr/src/results/mnist_smoothing_radius_sensitivity_vs_case_align_cosine_k25/mnist_smoothing_radius_plot_long_cosine_k25.csv
- Config: /Users/craigpirie/Documents/PhD/AGREE_ICCBR/agree-iccbr/src/results/mnist_smoothing_radius_sensitivity_vs_case_align_cosine_k25/mnist_smoothing_radius_config_cosine_k25.json
